# Classical baselines for 36-month MCI prognosis
Trains four strong conventional baselines on the existing ADNI MCI-prognosis folds:

1. regularised Logistic Regression;
2. RBF Support Vector Machine;
3. Random Forest;
4. XGBoost.

It is designed for the **final run**, not a minimal baseline.

## Experimental setup
- Reuse the existing five outer folds and existing `train` / `validation` / `test` roles.
- Never use outer-test participants for preprocessing, tuning, early stopping, model selection, or threshold selection.
- Use all **28 prepared non-MRI predictors**: 25 scaled continuous + 3 encoded categorical predictors.
- Preserve missingness through all **28 feature masks** and five non-MRI branch masks.
- Do not use raw MRI here. This is a *standard tabular ML baseline* experiment.
- One-hot encode the three categorical predictors rather than treating their encoded indices as continuous.
- Use broad but bounded hyperparameter searches rather than defaults.
- Rank candidates by 5-fold inner CV on the training subset, then expose only the top five to the fixed validation subset.
- Use seeds **17, 42, 73** for final stochastic fits.
- Keep the classification threshold fixed at **0.50**.
- Report ROC AUC (primary), average precision, balanced accuracy, sensitivity, specificity, F1, Brier score and 10-bin ECE.
- Save each fold/model/seed prediction immediately so a Colab disconnect does not destroy completed work.


In [ ]:
%pip -q install "xgboost>=2.0,<4"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from __future__ import annotations

import json
import platform
import sys
import time
import warnings
from pathlib import Path
from typing import Any, Dict, Tuple

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("/content/drive/MyDrive/adni_mri/models/3mt_tmc_evidential")
INPUT_ROOT = PROJECT_ROOT / "fold_ready_inputs_categorical_encoded" / "mci_prognosis"
MANIFEST_ROOT = PROJECT_ROOT / "manifests"

RUN_TAG = "standard_ml_baselines_mri_embeddings_reasonable_v1"
OUTPUT_ROOT = PROJECT_ROOT / "standard_ml_baselines" / RUN_TAG
SEARCH_RESULTS_DIR = OUTPUT_ROOT / "search_results"
PREDICTIONS_DIR = OUTPUT_ROOT / "fold_predictions"
TABLES_DIR = OUTPUT_ROOT / "tables"
METADATA_DIR = OUTPUT_ROOT / "metadata"
MODELS_DIR = OUTPUT_ROOT / "models"

for d in [OUTPUT_ROOT, SEARCH_RESULTS_DIR, PREDICTIONS_DIR, TABLES_DIR, METADATA_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RID_COL = "RID"
TARGET_COL = "MCI_PROGNOSIS_TARGET"
OUTER_FOLD_COL = "OUTER_FOLD"
ROLE_COL = "DATA_ROLE"

OUTER_FOLDS = [0, 1, 2, 3, 4]
FINAL_SEEDS = [17, 42, 73]
SEARCH_SEED = 17
THRESHOLD = 0.50

INNER_CV_SPLITS = 5
TOP_K_VALIDATION = 5
SEARCH_PROFILE = "reasonable"  # compute-efficient final search
N_JOBS = -1

SAVE_MODELS = False
RESUME_EXISTING_RUN = True

INCLUDE_FEATURE_MASKS = True
INCLUDE_NON_MRI_BRANCH_MASKS = True
INCLUDE_MRI_AVAILABILITY_MASK = True

# MRI embeddings are extracted separately from the already-trained
# fold-specific MRI encoder and then treated as ordinary numeric inputs here.
# The representation is fixed across the classifier seeds in this notebook.
MRI_EMBEDDING_ROOT = PROJECT_ROOT / "standard_ml_baselines" / "mri_embeddings"
MRI_EMBEDDING_SEED = SEARCH_SEED
MRI_EMBEDDING_DIM = 64
MRI_EMBEDDING_COLS = [
    f"MRI_EMB_{i:02d}" for i in range(MRI_EMBEDDING_DIM)
]

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("Search profile:", SEARCH_PROFILE)
print("Output root:", OUTPUT_ROOT)


## 1. Exact tabular predictor contract

The prepared MCI prognosis tables contain 25 scaled continuous predictors and 3 encoded categorical predictors.

The feature masks are used explicitly. If a feature is unavailable, its numerical placeholder is neutralised to zero while the corresponding mask remains available to the classifier. This prevents an unavailable value from being indistinguishable from a genuinely observed zero/mean-valued feature.

The five non-MRI branch masks are retained exactly as before.

The only added input is the **64-dimensional fold-specific MRI representation** extracted from the trained CNN–transformer MRI encoder. The MRI availability mask is included alongside these embeddings. No other baseline feature handling, tuning logic, folds, seeds, metrics, or evaluation code is changed. This version uses the compute-efficient `reasonable` hyperparameter budget defined below.


In [ ]:
CONTINUOUS_COLS = [
    "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
    "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
    "ADAS__TOTSCORE__Z",
    "ADAS__TOTAL13__Z",
    "MMSE__MMSE_TOTAL_SCORE__Z",
    "MMSE__MMSE_ORIENTATION_SCORE__Z",
    "MMSE__MMSE_REGISTRATION_SCORE__Z",
    "MMSE__MMSE_ATTENTION_SCORE__Z",
    "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
    "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
    "FAQ__FAQTOTAL__Z",
    "CSF__ABETA40__Z",
    "CSF__ABETA42__Z",
    "CSF__TAU__Z",
    "CSF__PTAU__Z",
    "CSF__ABETA42_40_RATIO__Z",
    "PLASMA__pT217_F__Z",
    "PLASMA__AB42_F__Z",
    "PLASMA__AB40_F__Z",
    "PLASMA__AB42_AB40_F__Z",
    "PLASMA__pT217_AB42_F__Z",
    "PLASMA__NfL_Q__Z",
    "PLASMA__GFAP_Q__Z",
    "PLASMA__NfL_F__Z",
    "PLASMA__GFAP_F__Z",
]

CATEGORICAL_COLS = [
    "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
    "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
    "APOE__APOE4_ALLELE_COUNT__IDX",
]

BRANCH_MASK_COLS = [
    "BRANCH_MASK__DEMOGRAPHICS",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    "BRANCH_MASK__CSF",
    "BRANCH_MASK__PLASMA",
    "BRANCH_MASK__APOE",
]

MRI_BRANCH_MASK_COL = "BRANCH_MASK__MRI"
TABULAR_PREDICTOR_COLS = CONTINUOUS_COLS + CATEGORICAL_COLS

assert len(CONTINUOUS_COLS) == 25
assert len(CATEGORICAL_COLS) == 3
assert len(TABULAR_PREDICTOR_COLS) == 28

def predictor_to_feature_mask(predictor: str) -> str:
    base = predictor
    for suffix in ("__Z", "__IDX"):
        if base.endswith(suffix):
            base = base[:-len(suffix)]
            break
    return "FEATURE_MASK__" + base.replace("__", "_")

EXPECTED_FEATURE_MASKS = [predictor_to_feature_mask(c) for c in TABULAR_PREDICTOR_COLS]
assert len(EXPECTED_FEATURE_MASKS) == 28
assert len(set(EXPECTED_FEATURE_MASKS)) == 28

print("Continuous:", len(CONTINUOUS_COLS))
print("Categorical:", len(CATEGORICAL_COLS))
print("Feature masks:", len(EXPECTED_FEATURE_MASKS))


In [ ]:
def load_json_if_exists(path: Path):
    if not path.is_file():
        print(f"Manifest not found; CSV validation will still run: {path}")
        return None
    with open(path, "r") as f:
        return json.load(f)

feature_mask_meta = load_json_if_exists(MANIFEST_ROOT / "feature_mask_columns.json")
branch_mask_meta = load_json_if_exists(MANIFEST_ROOT / "branch_mask_columns.json")
target_mapping_meta = load_json_if_exists(MANIFEST_ROOT / "task_target_mappings.json")

print("Feature-mask manifest:", feature_mask_meta is not None)
print("Branch-mask manifest:", branch_mask_meta is not None)
print("Target-mapping manifest:", target_mapping_meta is not None)


## 2. Data validation and leakage safeguards

Input validation fails if the cohort size, split counts, target mapping, masks, or prepared values are inconsistent.

The existing fold-specific scaling and categorical encoding are retained. No statistics are recalculated from validation or test participants.


In [ ]:
EXPECTED_ROLE_COUNTS = {
    0: {"train": 369, "validation": 66, "test": 109},
    1: {"train": 369, "validation": 66, "test": 109},
    2: {"train": 369, "validation": 66, "test": 109},
    3: {"train": 369, "validation": 66, "test": 109},
    4: {"train": 370, "validation": 66, "test": 108},
}

def fold_path(fold: int) -> Path:
    return INPUT_ROOT / f"mci_prognosis_outer_fold_{fold}_categorical_encoded.csv"

def _coerce_binary_mask(s: pd.Series, name: str) -> pd.Series:
    out = pd.to_numeric(s, errors="raise").astype(int)
    bad = set(out.unique()) - {0, 1}
    if bad:
        raise ValueError(f"{name} contains values outside 0/1: {sorted(bad)}")
    return out

def load_and_validate_fold(fold: int) -> pd.DataFrame:
    path = fold_path(fold)
    if not path.is_file():
        raise FileNotFoundError(f"Missing prepared fold file: {path}")

    df = pd.read_csv(path)

    required = {
        RID_COL, TARGET_COL, OUTER_FOLD_COL, ROLE_COL,
        *TABULAR_PREDICTOR_COLS,
        *EXPECTED_FEATURE_MASKS,
        *BRANCH_MASK_COLS,
        MRI_BRANCH_MASK_COL,
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Fold {fold}: required columns missing:\n{missing}")

    if len(df) != 544:
        raise ValueError(f"Fold {fold}: expected 544 rows, found {len(df)}")
    if df[RID_COL].duplicated().any():
        raise ValueError(f"Fold {fold}: duplicate RID values found.")

    df[RID_COL] = pd.to_numeric(df[RID_COL], errors="raise").astype(int)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="raise").astype(int)

    if set(df[TARGET_COL].unique()) != {0, 1}:
        raise ValueError(f"Fold {fold}: expected binary targets {{0,1}}.")

    df[ROLE_COL] = df[ROLE_COL].astype(str).str.strip().str.lower()
    actual_counts = df[ROLE_COL].value_counts().to_dict()

    for role, expected_n in EXPECTED_ROLE_COUNTS[fold].items():
        actual_n = int(actual_counts.get(role, 0))
        if actual_n != expected_n:
            raise ValueError(
                f"Fold {fold}: {role} expected {expected_n}, found {actual_n}"
            )

    unexpected_roles = set(actual_counts) - {"train", "validation", "test"}
    if unexpected_roles:
        raise ValueError(f"Fold {fold}: unexpected DATA_ROLE values {unexpected_roles}")

    test_outer = pd.to_numeric(
        df.loc[df[ROLE_COL] == "test", OUTER_FOLD_COL], errors="raise"
    ).astype(int)
    if not (test_outer == fold).all():
        raise ValueError(f"Fold {fold}: some test rows do not carry OUTER_FOLD={fold}")

    for c in EXPECTED_FEATURE_MASKS + BRANCH_MASK_COLS + [MRI_BRANCH_MASK_COL]:
        df[c] = _coerce_binary_mask(df[c], c)

    for predictor, mask_col in zip(TABULAR_PREDICTOR_COLS, EXPECTED_FEATURE_MASKS):
        numeric = pd.to_numeric(df[predictor], errors="coerce")

        observed_but_nan = (df[mask_col] == 1) & numeric.isna()
        if observed_but_nan.any():
            rids = df.loc[observed_but_nan, RID_COL].head().tolist()
            raise ValueError(
                f"Fold {fold}: {predictor} is observed by mask but NaN for RIDs {rids}"
            )

        numeric = numeric.fillna(0.0)
        numeric.loc[df[mask_col] == 0] = 0.0
        df[predictor] = numeric

    for predictor in CATEGORICAL_COLS:
        mask_col = predictor_to_feature_mask(predictor)
        df[predictor] = pd.to_numeric(df[predictor], errors="raise").astype(int)
        df.loc[df[mask_col] == 0, predictor] = 0

    return df

fold_tables = {fold: load_and_validate_fold(fold) for fold in OUTER_FOLDS}

rows = []
for fold, df in fold_tables.items():
    counts = df[ROLE_COL].value_counts()
    rows.append({
        "fold": fold,
        "n_total": len(df),
        "n_train": int(counts["train"]),
        "n_validation": int(counts["validation"]),
        "n_test": int(counts["test"]),
        "n_sMCI": int((df[TARGET_COL] == 0).sum()),
        "n_pMCI": int((df[TARGET_COL] == 1).sum()),
    })
display(pd.DataFrame(rows))


In [ ]:
def _infer_current_outer_fold(df: pd.DataFrame) -> int:
    test_folds = (
        pd.to_numeric(
            df.loc[df[ROLE_COL] == "test", OUTER_FOLD_COL],
            errors="raise",
        )
        .astype(int)
        .unique()
    )
    if len(test_folds) != 1:
        raise ValueError(
            "Could not infer exactly one current outer fold from the test rows: "
            f"{test_folds.tolist()}"
        )
    return int(test_folds[0])

def mri_embedding_path(fold: int) -> Path:
    return (
        MRI_EMBEDDING_ROOT
        / f"mci_prognosis_fold_{fold}_seed_{MRI_EMBEDDING_SEED}_mri_embeddings.csv"
    )

def load_and_validate_mri_embeddings(
    fold: int,
    fold_df: pd.DataFrame,
) -> pd.DataFrame:
    path = mri_embedding_path(fold)
    if not path.is_file():
        raise FileNotFoundError(
            "Missing MRI embedding file. Run the MRI-embedding extraction "
            f"step first:\n{path}"
        )

    emb = pd.read_csv(path)

    required = {
        RID_COL,
        MRI_BRANCH_MASK_COL,
        *MRI_EMBEDDING_COLS,
    }
    missing = sorted(required - set(emb.columns))
    if missing:
        raise ValueError(
            f"Fold {fold}: MRI embedding file is missing required columns:\n"
            f"{missing}"
        )

    if len(emb) != 544:
        raise ValueError(
            f"Fold {fold}: expected 544 MRI-embedding rows, found {len(emb)}"
        )

    emb[RID_COL] = pd.to_numeric(emb[RID_COL], errors="raise").astype(int)
    if emb[RID_COL].duplicated().any():
        raise ValueError(
            f"Fold {fold}: duplicate RID values in MRI embedding file."
        )

    emb[MRI_BRANCH_MASK_COL] = _coerce_binary_mask(
        emb[MRI_BRANCH_MASK_COL],
        MRI_BRANCH_MASK_COL,
    )

    for col in MRI_EMBEDDING_COLS:
        emb[col] = pd.to_numeric(emb[col], errors="raise").astype(float)

    embedding_values = emb[MRI_EMBEDDING_COLS].to_numpy(dtype=np.float64)
    if not np.isfinite(embedding_values).all():
        raise ValueError(
            f"Fold {fold}: MRI embeddings contain NaN or infinite values."
        )

    base_ids = set(fold_df[RID_COL].astype(int))
    embedding_ids = set(emb[RID_COL].astype(int))
    if base_ids != embedding_ids:
        missing_from_embeddings = sorted(base_ids - embedding_ids)
        extra_in_embeddings = sorted(embedding_ids - base_ids)
        raise ValueError(
            f"Fold {fold}: RID mismatch between prepared fold table and MRI "
            f"embeddings. Missing={missing_from_embeddings[:10]}, "
            f"extra={extra_in_embeddings[:10]}"
        )

    mask_check = (
        fold_df[[RID_COL, MRI_BRANCH_MASK_COL]]
        .merge(
            emb[[RID_COL, MRI_BRANCH_MASK_COL]],
            on=RID_COL,
            how="inner",
            validate="one_to_one",
            suffixes=("_prepared", "_embedding"),
        )
    )

    if not (
        mask_check[f"{MRI_BRANCH_MASK_COL}_prepared"].astype(int)
        == mask_check[f"{MRI_BRANCH_MASK_COL}_embedding"].astype(int)
    ).all():
        raise ValueError(
            f"Fold {fold}: MRI availability masks disagree between the "
            "prepared fold table and extracted MRI embeddings."
        )

    unavailable = emb[MRI_BRANCH_MASK_COL].to_numpy(dtype=int) == 0
    if np.any(unavailable):
        unavailable_values = embedding_values[unavailable]
        if not np.allclose(unavailable_values, 0.0, atol=1e-8):
            raise ValueError(
                f"Fold {fold}: masked MRI embeddings must be exactly zero "
                "when MRI is unavailable."
            )

    return emb[[RID_COL, *MRI_EMBEDDING_COLS]].copy()

def make_outer_fold_matrices(df: pd.DataFrame):
    fold = _infer_current_outer_fold(df)
    mri_embeddings = load_and_validate_mri_embeddings(
        fold=fold,
        fold_df=df,
    )

    df = df.merge(
        mri_embeddings,
        on=RID_COL,
        how="left",
        validate="one_to_one",
    )

    train = df.loc[df[ROLE_COL] == "train"].copy()
    val = df.loc[df[ROLE_COL] == "validation"].copy()
    test = df.loc[df[ROLE_COL] == "test"].copy()

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float64)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float64)

    ohe.fit(train[CATEGORICAL_COLS])

    def transform(part: pd.DataFrame):
        x_cont = part[CONTINUOUS_COLS].to_numpy(dtype=np.float64)
        x_cat = ohe.transform(part[CATEGORICAL_COLS])
        x_mri = part[MRI_EMBEDDING_COLS].to_numpy(dtype=np.float64)

        extra_cols = []
        if INCLUDE_FEATURE_MASKS:
            extra_cols.extend(EXPECTED_FEATURE_MASKS)
        if INCLUDE_NON_MRI_BRANCH_MASKS:
            extra_cols.extend(BRANCH_MASK_COLS)
        if INCLUDE_MRI_AVAILABILITY_MASK:
            extra_cols.append(MRI_BRANCH_MASK_COL)

        pieces = [x_cont, x_cat, x_mri]
        if extra_cols:
            pieces.append(part[extra_cols].to_numpy(dtype=np.float64))

        X = np.concatenate(pieces, axis=1)
        y = part[TARGET_COL].to_numpy(dtype=int)
        rid = part[RID_COL].to_numpy(dtype=int)
        return X, y, rid

    X_train, y_train, rid_train = transform(train)
    X_val, y_val, rid_val = transform(val)
    X_test, y_test, rid_test = transform(test)

    feature_names = list(CONTINUOUS_COLS)
    feature_names += list(ohe.get_feature_names_out(CATEGORICAL_COLS))
    feature_names += MRI_EMBEDDING_COLS
    if INCLUDE_FEATURE_MASKS:
        feature_names += EXPECTED_FEATURE_MASKS
    if INCLUDE_NON_MRI_BRANCH_MASKS:
        feature_names += BRANCH_MASK_COLS
    if INCLUDE_MRI_AVAILABILITY_MASK:
        feature_names += [MRI_BRANCH_MASK_COL]

    assert X_train.shape[1] == len(feature_names)

    return {
        "X_train": X_train, "y_train": y_train, "rid_train": rid_train,
        "X_val": X_val, "y_val": y_val, "rid_val": rid_val,
        "X_test": X_test, "y_test": y_test, "rid_test": rid_test,
        "ohe": ohe,
        "feature_names": feature_names,
    }

example = make_outer_fold_matrices(fold_tables[0])
print("Fold 0 train:", example["X_train"].shape)
print("Fold 0 validation:", example["X_val"].shape)
print("Fold 0 test:", example["X_test"].shape)
print("MRI embedding source:", mri_embedding_path(0))
print("Transformed feature count:", len(example["feature_names"]))


## 3. Evaluation metrics

Threshold-dependent metrics use the fixed 0.50 pMCI threshold. ECE uses 10 equal-width probability bins.


In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10) -> float:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, edges[1:-1], right=False)

    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        idx = bin_ids == b
        if not np.any(idx):
            continue
        observed_rate = float(np.mean(y_true[idx]))
        mean_prob = float(np.mean(y_prob[idx]))
        ece += (np.sum(idx) / n) * abs(observed_rate - mean_prob)
    return float(ece)

def classification_metrics(y_true, y_prob, threshold=THRESHOLD) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "average_precision": float(average_precision_score(y_true, y_prob)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "ece_10bin": float(expected_calibration_error(y_true, y_prob, 10)),
        "n": int(len(y_true)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
    }

def ranking_metrics(y_true, scores):
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }


## 4. Search budgets

The default `reasonable` profile is intended for the final baseline run while keeping the search proportionate to this cohort size.

- **Logistic Regression:** L2 + elastic-net, five regularisation strengths, three elastic-net mixing values, and both unweighted/balanced class weighting. This is **40 configurations × 5 inner folds = 200 CV fits per outer fold**, instead of 910.
- **RBF SVM:** four `C` values × four `gamma` choices × two class-weight options = **32 configurations × 5 = 160 CV fits per outer fold**, instead of 1,430.
- **Random Forest:** 30 randomized configurations with 300 trees during search; selected model refit with 1,000 trees.
- **XGBoost:** 30 randomized configurations with 750 trees during inner search; top candidates are then refit with up to 3,000 trees and validation-based early stopping.

The five-fold inner CV, top-five validation selection, five outer folds, final seeds, metrics, threshold, feature set and missingness handling are unchanged.

The `thorough` profile is retained if a much larger search is ever needed. The `debug` profile remains execution-testing only.


In [ ]:
if SEARCH_PROFILE == "thorough":
    LR_C_VALUES = np.logspace(-4, 4, 13).tolist()
    LR_L1_RATIOS = [0.10, 0.25, 0.50, 0.75, 0.90, 1.00]
    LR_MAX_ITER = 20000
    LR_TOL = 1e-5

    SVM_C_VALUES = np.logspace(-3, 3, 13).tolist()
    SVM_GAMMA_VALUES = np.logspace(-4, 1, 11).tolist()

    RF_RANDOM_CANDIDATES = 80
    XGB_RANDOM_CANDIDATES = 80
    RF_SEARCH_TREES = 500
    RF_FINAL_TREES = 1500
    XGB_SEARCH_TREES = 1500
    XGB_FINAL_MAX_TREES = 5000
    XGB_EARLY_STOPPING_ROUNDS = 100

elif SEARCH_PROFILE == "reasonable":
    LR_C_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0]
    LR_L1_RATIOS = [0.25, 0.50, 0.75]
    LR_MAX_ITER = 5000
    LR_TOL = 1e-4

    SVM_C_VALUES = [0.1, 1.0, 10.0, 100.0]
    SVM_GAMMA_VALUES = ["scale", 0.001, 0.01, 0.1]

    RF_RANDOM_CANDIDATES = 30
    XGB_RANDOM_CANDIDATES = 30
    RF_SEARCH_TREES = 300
    RF_FINAL_TREES = 1000
    XGB_SEARCH_TREES = 750
    XGB_FINAL_MAX_TREES = 3000
    XGB_EARLY_STOPPING_ROUNDS = 75

elif SEARCH_PROFILE == "debug":
    LR_C_VALUES = [0.01, 0.1, 1.0, 10.0]
    LR_L1_RATIOS = [0.25, 0.50, 0.75]
    LR_MAX_ITER = 2000
    LR_TOL = 1e-4

    SVM_C_VALUES = [0.1, 1.0, 10.0]
    SVM_GAMMA_VALUES = [0.001, 0.01, 0.1]

    RF_RANDOM_CANDIDATES = 8
    XGB_RANDOM_CANDIDATES = 8
    RF_SEARCH_TREES = 100
    RF_FINAL_TREES = 250
    XGB_SEARCH_TREES = 300
    XGB_FINAL_MAX_TREES = 800
    XGB_EARLY_STOPPING_ROUNDS = 30

else:
    raise ValueError(
        "SEARCH_PROFILE must be 'thorough', 'reasonable', or 'debug'."
    )

INNER_CV = StratifiedKFold(
    n_splits=INNER_CV_SPLITS,
    shuffle=True,
    random_state=SEARCH_SEED,
)
SCORING = {"roc_auc": "roc_auc", "ap": "average_precision"}

lr_candidates = (
    len(LR_C_VALUES) * 2
    + len(LR_C_VALUES) * len(LR_L1_RATIOS) * 2
)
svm_candidates = (
    len(SVM_C_VALUES) * len(SVM_GAMMA_VALUES) * 2
)

print("Logistic grid candidates:", lr_candidates)
print("Logistic inner-CV fits:", lr_candidates * INNER_CV_SPLITS)
print("SVM grid candidates:", svm_candidates)
print("SVM inner-CV fits:", svm_candidates * INNER_CV_SPLITS)
print("RF randomized candidates:", RF_RANDOM_CANDIDATES)
print("XGB randomized candidates:", XGB_RANDOM_CANDIDATES)


In [ ]:
def json_safe(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj

def save_json(path: Path, data: Any):
    with open(path, "w") as f:
        json.dump(json_safe(data), f, indent=2, sort_keys=True)

def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def save_cv_results(path: Path, cv_results: Dict[str, Any]):
    df = pd.DataFrame(cv_results).copy()
    if "params" in df.columns:
        df["params_json"] = df["params"].apply(
            lambda p: json.dumps(json_safe(p), sort_keys=True)
        )
        df = df.drop(columns=["params"])
    df.to_csv(path, index=False)

def top_candidates(cv_results: Dict[str, Any], top_k=TOP_K_VALIDATION):
    df = pd.DataFrame(cv_results).copy()
    return (
        df.sort_values(
            ["mean_test_roc_auc", "mean_test_ap"],
            ascending=[False, False],
        )
        .head(top_k)
        .reset_index(drop=True)
    )

def validation_select(top_df, build_model_fn, X_train, y_train, X_val, y_val):
    rows = []
    for rank, row in top_df.iterrows():
        params = dict(row["params"])
        model = build_model_fn(params)
        model.fit(X_train, y_train)

        if hasattr(model, "predict_proba"):
            scores = model.predict_proba(X_val)[:, 1]
        else:
            scores = model.decision_function(X_val)

        m = ranking_metrics(y_val, scores)
        rows.append({
            "inner_cv_rank": rank + 1,
            "inner_cv_roc_auc": float(row["mean_test_roc_auc"]),
            "inner_cv_ap": float(row["mean_test_ap"]),
            "validation_roc_auc": m["roc_auc"],
            "validation_ap": m["average_precision"],
            "params": params,
        })

    ranking = pd.DataFrame(rows).sort_values(
        ["validation_roc_auc", "validation_ap", "inner_cv_roc_auc"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    return dict(ranking.iloc[0]["params"]), ranking

def save_validation_ranking(path: Path, ranking: pd.DataFrame):
    out = ranking.copy()
    out["params_json"] = out["params"].apply(
        lambda p: json.dumps(json_safe(p), sort_keys=True)
    )
    out.drop(columns=["params"]).to_csv(path, index=False)


In [ ]:
def tune_logistic(fold, X_train, y_train, X_val, y_val):
    selected_path = SEARCH_RESULTS_DIR / f"fold_{fold}_logistic_selected.json"
    if RESUME_EXISTING_RUN and selected_path.is_file():
        print(f"[resume] fold {fold} logistic")
        return load_json(selected_path)

    base = LogisticRegression(
        solver="saga",
        max_iter=LR_MAX_ITER,
        tol=LR_TOL,
        random_state=SEARCH_SEED,
    )

    grid = [
        {
            "penalty": ["l2"],
            "C": LR_C_VALUES,
            "class_weight": [None, "balanced"],
        },
        {
            "penalty": ["elasticnet"],
            "C": LR_C_VALUES,
            "l1_ratio": LR_L1_RATIOS,
            "class_weight": [None, "balanced"],
        },
    ]

    search = GridSearchCV(
        base,
        grid,
        scoring=SCORING,
        refit=False,
        cv=INNER_CV,
        n_jobs=N_JOBS,
        error_score="raise",
    )
    search.fit(X_train, y_train)

    save_cv_results(
        SEARCH_RESULTS_DIR / f"fold_{fold}_logistic_inner_cv.csv",
        search.cv_results_,
    )

    top = top_candidates(search.cv_results_)

    def builder(params):
        return LogisticRegression(
            solver="saga",
            max_iter=20000,
            tol=1e-5,
            random_state=SEARCH_SEED,
            **params,
        )

    best_params, ranking = validation_select(
        top, builder, X_train, y_train, X_val, y_val
    )
    save_validation_ranking(
        SEARCH_RESULTS_DIR / f"fold_{fold}_logistic_validation_ranking.csv",
        ranking,
    )

    payload = {
        "model": "logistic_regression",
        "fold": fold,
        "selected_params": best_params,
        "search_profile": SEARCH_PROFILE,
    }
    save_json(selected_path, payload)
    return payload


In [ ]:
def tune_svm(fold, X_train, y_train, X_val, y_val):
    selected_path = SEARCH_RESULTS_DIR / f"fold_{fold}_svm_selected.json"
    if RESUME_EXISTING_RUN and selected_path.is_file():
        print(f"[resume] fold {fold} svm")
        return load_json(selected_path)

    base = SVC(kernel="rbf", probability=False, cache_size=2000)
    grid = {
        "C": SVM_C_VALUES,
        "gamma": SVM_GAMMA_VALUES,
        "class_weight": [None, "balanced"],
    }

    search = GridSearchCV(
        base,
        grid,
        scoring=SCORING,
        refit=False,
        cv=INNER_CV,
        n_jobs=N_JOBS,
        error_score="raise",
    )
    search.fit(X_train, y_train)

    save_cv_results(
        SEARCH_RESULTS_DIR / f"fold_{fold}_svm_inner_cv.csv",
        search.cv_results_,
    )

    top = top_candidates(search.cv_results_)

    def builder(params):
        return SVC(
            kernel="rbf",
            probability=False,
            cache_size=2000,
            **params,
        )

    best_params, ranking = validation_select(
        top, builder, X_train, y_train, X_val, y_val
    )
    save_validation_ranking(
        SEARCH_RESULTS_DIR / f"fold_{fold}_svm_validation_ranking.csv",
        ranking,
    )

    payload = {
        "model": "svm_rbf",
        "fold": fold,
        "selected_params": best_params,
        "search_profile": SEARCH_PROFILE,
    }
    save_json(selected_path, payload)
    return payload


In [ ]:
def tune_random_forest(fold, X_train, y_train, X_val, y_val):
    selected_path = SEARCH_RESULTS_DIR / f"fold_{fold}_random_forest_selected.json"
    if RESUME_EXISTING_RUN and selected_path.is_file():
        print(f"[resume] fold {fold} random forest")
        return load_json(selected_path)

    base = RandomForestClassifier(
        n_estimators=RF_SEARCH_TREES,
        random_state=SEARCH_SEED,
        n_jobs=1,
    )

    distributions = {
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 3, 4, 5, 6, 8, 10, 14],
        "min_samples_split": [2, 4, 6, 10, 16],
        "min_samples_leaf": [1, 2, 3, 4, 6, 8, 12],
        "max_features": ["sqrt", "log2", 0.30, 0.50, 0.70, 1.0],
        "class_weight": [None, "balanced", "balanced_subsample"],
        "bootstrap": [True],
        "max_samples": [None, 0.70, 0.85, 1.0],
    }

    search = RandomizedSearchCV(
        base,
        distributions,
        n_iter=RF_RANDOM_CANDIDATES,
        scoring=SCORING,
        refit=False,
        cv=INNER_CV,
        random_state=SEARCH_SEED,
        n_jobs=N_JOBS,
        error_score="raise",
    )
    search.fit(X_train, y_train)

    save_cv_results(
        SEARCH_RESULTS_DIR / f"fold_{fold}_random_forest_inner_cv.csv",
        search.cv_results_,
    )

    top = top_candidates(search.cv_results_)

    def builder(params):
        return RandomForestClassifier(
            n_estimators=RF_SEARCH_TREES,
            random_state=SEARCH_SEED,
            n_jobs=N_JOBS,
            **params,
        )

    best_params, ranking = validation_select(
        top, builder, X_train, y_train, X_val, y_val
    )
    save_validation_ranking(
        SEARCH_RESULTS_DIR / f"fold_{fold}_random_forest_validation_ranking.csv",
        ranking,
    )

    payload = {
        "model": "random_forest",
        "fold": fold,
        "selected_params": best_params,
        "search_profile": SEARCH_PROFILE,
        "search_trees": RF_SEARCH_TREES,
        "final_trees": RF_FINAL_TREES,
    }
    save_json(selected_path, payload)
    return payload


In [ ]:
def make_xgb(seed, n_estimators, early_stopping_rounds=None, **params):
    kwargs = dict(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_estimators=n_estimators,
        random_state=seed,
        n_jobs=1,
        verbosity=0,
    )
    kwargs.update(params)
    if early_stopping_rounds is not None:
        kwargs["early_stopping_rounds"] = early_stopping_rounds
    return xgb.XGBClassifier(**kwargs)

def fit_xgb_early_stop(model, X_train, y_train, X_val, y_val):
    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        return model
    except TypeError:
        params = model.get_params()
        esr = params.pop("early_stopping_rounds", XGB_EARLY_STOPPING_ROUNDS)
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
            early_stopping_rounds=esr,
        )
        return model

def tune_xgboost(fold, X_train, y_train, X_val, y_val):
    selected_path = SEARCH_RESULTS_DIR / f"fold_{fold}_xgboost_selected.json"
    if RESUME_EXISTING_RUN and selected_path.is_file():
        print(f"[resume] fold {fold} xgboost")
        return load_json(selected_path)

    neg = int(np.sum(y_train == 0))
    pos = int(np.sum(y_train == 1))
    balanced_ratio = neg / pos

    base = make_xgb(SEARCH_SEED, XGB_SEARCH_TREES)

    distributions = {
        "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.10],
        "max_depth": [2, 3, 4, 5, 6],
        "min_child_weight": [1, 2, 3, 5, 8, 12],
        "subsample": [0.60, 0.75, 0.90, 1.00],
        "colsample_bytree": [0.60, 0.75, 0.90, 1.00],
        "gamma": [0.0, 0.05, 0.10, 0.25, 0.50],
        "reg_alpha": [0.0, 1e-4, 1e-3, 1e-2, 0.10, 0.50, 1.0],
        "reg_lambda": [0.10, 0.50, 1.0, 2.0, 5.0, 10.0, 20.0],
        "scale_pos_weight": [1.0, float(balanced_ratio)],
    }

    search = RandomizedSearchCV(
        base,
        distributions,
        n_iter=XGB_RANDOM_CANDIDATES,
        scoring=SCORING,
        refit=False,
        cv=INNER_CV,
        random_state=SEARCH_SEED,
        n_jobs=N_JOBS,
        error_score="raise",
    )
    search.fit(X_train, y_train)

    save_cv_results(
        SEARCH_RESULTS_DIR / f"fold_{fold}_xgboost_inner_cv.csv",
        search.cv_results_,
    )

    top = top_candidates(search.cv_results_)
    rows = []

    for rank, row in top.iterrows():
        params = dict(row["params"])
        model = make_xgb(
            SEARCH_SEED,
            XGB_FINAL_MAX_TREES,
            XGB_EARLY_STOPPING_ROUNDS,
            **params,
        )
        model = fit_xgb_early_stop(model, X_train, y_train, X_val, y_val)

        prob = model.predict_proba(X_val)[:, 1]
        m = ranking_metrics(y_val, prob)

        rows.append({
            "inner_cv_rank": rank + 1,
            "inner_cv_roc_auc": float(row["mean_test_roc_auc"]),
            "inner_cv_ap": float(row["mean_test_ap"]),
            "validation_roc_auc": m["roc_auc"],
            "validation_ap": m["average_precision"],
            "best_iteration": getattr(model, "best_iteration", None),
            "params": params,
        })

    ranking = pd.DataFrame(rows).sort_values(
        ["validation_roc_auc", "validation_ap", "inner_cv_roc_auc"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    save_validation_ranking(
        SEARCH_RESULTS_DIR / f"fold_{fold}_xgboost_validation_ranking.csv",
        ranking,
    )

    payload = {
        "model": "xgboost",
        "fold": fold,
        "selected_params": dict(ranking.iloc[0]["params"]),
        "search_profile": SEARCH_PROFILE,
        "search_trees": XGB_SEARCH_TREES,
        "final_max_trees": XGB_FINAL_MAX_TREES,
        "early_stopping_rounds": XGB_EARLY_STOPPING_ROUNDS,
        "search_seed_best_iteration": ranking.iloc[0]["best_iteration"],
    }
    save_json(selected_path, payload)
    return payload


## 5. Final fitting across seeds

Hyperparameter selection happens once per outer fold using search seed 17. The selected configuration is then refitted independently with seeds 17, 42, and 73.

This measures stochastic variation without giving every seed a separate hyperparameter lottery.


In [ ]:
MODEL_NAMES = [
    "logistic_regression",
    "svm_rbf",
    "random_forest",
    "xgboost",
]

def prediction_path(model_name, fold, seed):
    return PREDICTIONS_DIR / f"{model_name}_fold_{fold}_seed_{seed}_test_predictions.csv"

def build_final_non_xgb(model_name, payload, seed):
    params = dict(payload["selected_params"])

    if model_name == "logistic_regression":
        return LogisticRegression(
            solver="saga",
            max_iter=LR_MAX_ITER,
            tol=LR_TOL,
            random_state=seed,
            **params,
        )

    if model_name == "svm_rbf":
        return SVC(
            kernel="rbf",
            probability=True,
            cache_size=2000,
            random_state=seed,
            **params,
        )

    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=RF_FINAL_TREES,
            random_state=seed,
            n_jobs=N_JOBS,
            **params,
        )

    raise ValueError(model_name)

def save_predictions(path, model_name, fold, seed, rids, y_true, y_prob):
    out = pd.DataFrame({
        "MODEL": model_name,
        "SEED": int(seed),
        "FOLD": int(fold),
        "RID": np.asarray(rids, dtype=int),
        "TARGET": np.asarray(y_true, dtype=int),
        "PROB_PMCI": np.asarray(y_prob, dtype=float),
    })
    out["PREDICTED_CLASS_0P5"] = (out["PROB_PMCI"] >= THRESHOLD).astype(int)

    if out["RID"].duplicated().any():
        raise ValueError("Duplicate RIDs inside one test prediction file.")
    out.to_csv(path, index=False)

def tune_all_for_fold(fold, M):
    X_train, y_train = M["X_train"], M["y_train"]
    X_val, y_val = M["X_val"], M["y_val"]

    selected = {}
    print(f"\nFold {fold}: tuning Logistic Regression")
    selected["logistic_regression"] = tune_logistic(
        fold, X_train, y_train, X_val, y_val
    )

    print(f"Fold {fold}: tuning RBF SVM")
    selected["svm_rbf"] = tune_svm(
        fold, X_train, y_train, X_val, y_val
    )

    print(f"Fold {fold}: tuning Random Forest")
    selected["random_forest"] = tune_random_forest(
        fold, X_train, y_train, X_val, y_val
    )

    print(f"Fold {fold}: tuning XGBoost")
    selected["xgboost"] = tune_xgboost(
        fold, X_train, y_train, X_val, y_val
    )

    return selected

def fit_final_for_fold(fold, M, selected):
    X_train, y_train = M["X_train"], M["y_train"]
    X_val, y_val = M["X_val"], M["y_val"]
    X_test, y_test, rid_test = M["X_test"], M["y_test"], M["rid_test"]

    for model_name in MODEL_NAMES:
        for seed in FINAL_SEEDS:
            path = prediction_path(model_name, fold, seed)

            if RESUME_EXISTING_RUN and path.is_file():
                print(f"[resume] exists: {path.name}")
                continue

            print(f"Final fit: fold={fold} model={model_name} seed={seed}")

            if model_name == "xgboost":
                params = dict(selected[model_name]["selected_params"])
                model = make_xgb(
                    seed,
                    XGB_FINAL_MAX_TREES,
                    XGB_EARLY_STOPPING_ROUNDS,
                    **params,
                )
                model = fit_xgb_early_stop(
                    model, X_train, y_train, X_val, y_val
                )
            else:
                model = build_final_non_xgb(
                    model_name, selected[model_name], seed
                )
                model.fit(X_train, y_train)

            prob = model.predict_proba(X_test)[:, 1]
            save_predictions(
                path, model_name, fold, seed, rid_test, y_test, prob
            )

            if SAVE_MODELS:
                joblib.dump(
                    model,
                    MODELS_DIR / f"{model_name}_fold_{fold}_seed_{seed}.joblib"
                )


## 6. Run the full experiment

Confirm that `SEARCH_PROFILE == "reasonable"` before treating this notebook's outputs as the intended compute-efficient paper baseline.

 Completed model searches and fold/model/seed predictions are reused on rerun.


In [ ]:
if SEARCH_PROFILE == "debug":
    warnings.warn(
        "SEARCH_PROFILE is 'debug'. Do not use these outputs as final paper results."
    )

start = time.time()
feature_records = []

for fold in OUTER_FOLDS:
    print("\n" + "=" * 80)
    print(f"OUTER FOLD {fold}")
    print("=" * 80)

    M = make_outer_fold_matrices(fold_tables[fold])

    feature_records.append({
        "fold": fold,
        "n_transformed_features": len(M["feature_names"]),
        "feature_names": M["feature_names"],
    })

    selected = tune_all_for_fold(fold, M)
    fit_final_for_fold(fold, M, selected)

save_json(METADATA_DIR / "transformed_feature_names_by_fold.json", feature_records)

print(f"\nElapsed: {(time.time() - start) / 60:.1f} minutes")


## 7. Aggregate held-out predictions

For each model and seed, the five test folds must combine to exactly 544 unique participants: 295 sMCI and 249 pMCI.


In [ ]:
all_prediction_frames = []
fold_metric_rows = []

for model_name in MODEL_NAMES:
    for seed in FINAL_SEEDS:
        seed_frames = []

        for fold in OUTER_FOLDS:
            path = prediction_path(model_name, fold, seed)
            if not path.is_file():
                raise FileNotFoundError(f"Missing prediction file: {path}")

            p = pd.read_csv(path)
            seed_frames.append(p)

            m = classification_metrics(
                p["TARGET"].to_numpy(),
                p["PROB_PMCI"].to_numpy(),
            )
            fold_metric_rows.append({
                "model": model_name,
                "seed": seed,
                "fold": fold,
                **m,
            })

        pooled = pd.concat(seed_frames, ignore_index=True)

        if len(pooled) != 544:
            raise ValueError(
                f"{model_name} seed {seed}: expected 544 rows, found {len(pooled)}"
            )
        if pooled["RID"].nunique() != 544:
            raise ValueError(
                f"{model_name} seed {seed}: expected 544 unique RIDs, "
                f"found {pooled['RID'].nunique()}"
            )

        counts = pooled["TARGET"].value_counts().to_dict()
        if counts.get(0, 0) != 295 or counts.get(1, 0) != 249:
            raise ValueError(
                f"{model_name} seed {seed}: unexpected class counts {counts}"
            )

        pooled.to_csv(
            PREDICTIONS_DIR / f"{model_name}_seed_{seed}_pooled_out_of_fold_predictions.csv",
            index=False,
        )
        all_prediction_frames.append(pooled)

fold_metrics = pd.DataFrame(fold_metric_rows)
fold_metrics.to_csv(TABLES_DIR / "fold_level_test_metrics.csv", index=False)

all_predictions = pd.concat(all_prediction_frames, ignore_index=True)
all_predictions.to_csv(
    PREDICTIONS_DIR / "all_baseline_out_of_fold_predictions.csv",
    index=False,
)

display(fold_metrics.head())


In [ ]:
pooled_rows = []

for model_name in MODEL_NAMES:
    for seed in FINAL_SEEDS:
        p = all_predictions.loc[
            (all_predictions["MODEL"] == model_name)
            & (all_predictions["SEED"] == seed)
        ].copy()

        m = classification_metrics(
            p["TARGET"].to_numpy(),
            p["PROB_PMCI"].to_numpy(),
        )
        pooled_rows.append({
            "model": model_name,
            "seed": seed,
            **m,
        })

pooled_metrics = pd.DataFrame(pooled_rows)
pooled_metrics.to_csv(
    TABLES_DIR / "pooled_out_of_fold_metrics_by_seed.csv",
    index=False,
)

display(pooled_metrics.sort_values(["model", "seed"]).reset_index(drop=True))


In [ ]:
metric_cols = [
    "roc_auc",
    "average_precision",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1",
    "brier",
    "ece_10bin",
]

summary_rows = []
for model_name, g in pooled_metrics.groupby("model"):
    row = {"model": model_name}
    for metric in metric_cols:
        row[f"{metric}_mean"] = float(g[metric].mean())
        row[f"{metric}_sd"] = float(g[metric].std(ddof=1))
    summary_rows.append(row)

seed_summary = pd.DataFrame(summary_rows)
seed_summary.to_csv(
    TABLES_DIR / "pooled_metrics_mean_sd_across_seeds.csv",
    index=False,
)

display(seed_summary.sort_values("roc_auc_mean", ascending=False).reset_index(drop=True))


## 8. Reproducibility record

The final cell records the exact software versions, search budget, seeds, feature/mask choices, and paths used for the run.


In [ ]:
metadata = {
    "run_tag": RUN_TAG,
    "search_profile": SEARCH_PROFILE,
    "project_root": str(PROJECT_ROOT),
    "input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "outer_folds": OUTER_FOLDS,
    "final_seeds": FINAL_SEEDS,
    "search_seed": SEARCH_SEED,
    "threshold": THRESHOLD,
    "inner_cv_splits": INNER_CV_SPLITS,
    "top_k_validation": TOP_K_VALIDATION,
    "include_feature_masks": INCLUDE_FEATURE_MASKS,
    "include_non_mri_branch_masks": INCLUDE_NON_MRI_BRANCH_MASKS,
    "include_mri_availability_mask": INCLUDE_MRI_AVAILABILITY_MASK,
    "mri_embedding_root": str(MRI_EMBEDDING_ROOT),
    "mri_embedding_seed": int(MRI_EMBEDDING_SEED),
    "mri_embedding_dim": int(MRI_EMBEDDING_DIM),
    "n_continuous_predictors": len(CONTINUOUS_COLS),
    "n_categorical_predictors": len(CATEGORICAL_COLS),
    "n_feature_masks": len(EXPECTED_FEATURE_MASKS),
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
    },
    "search_budget": {
        "lr_c_values": LR_C_VALUES,
        "lr_l1_ratios": LR_L1_RATIOS,
        "lr_max_iter": LR_MAX_ITER,
        "lr_tol": LR_TOL,
        "svm_c_values": SVM_C_VALUES,
        "svm_gamma_values": SVM_GAMMA_VALUES,
        "rf_random_candidates": RF_RANDOM_CANDIDATES,
        "rf_search_trees": RF_SEARCH_TREES,
        "rf_final_trees": RF_FINAL_TREES,
        "xgb_random_candidates": XGB_RANDOM_CANDIDATES,
        "xgb_search_trees": XGB_SEARCH_TREES,
        "xgb_final_max_trees": XGB_FINAL_MAX_TREES,
        "xgb_early_stopping_rounds": XGB_EARLY_STOPPING_ROUNDS,
    },
}

save_json(METADATA_DIR / "experiment_metadata.json", metadata)

print("Saved outputs to:", OUTPUT_ROOT)
print("Key metrics:", TABLES_DIR / "pooled_metrics_mean_sd_across_seeds.csv")
print("All OOF predictions:", PREDICTIONS_DIR / "all_baseline_out_of_fold_predictions.csv")


## Downstream comparison

This training notebook does **not** yet perform paired bootstrap comparisons against Hybrid / 3MT / TMC.

It saves the baseline out-of-fold probabilities in the exact form needed for a separate analysis notebook, where the analysis can:

- merge by RID / fold / seed;
- verify paired identities and targets;
- reproduce the thesis-style 2,000 paired participant-level bootstrap comparisons;
- create the final comparison table and figures.

It also does not average the three seeds into an ensemble for the primary result, because that would define a different model.
